# Meme Stock Signal Backtest
End-to-end: Reddit mention velocity → signal → backtest → results

In [ ]:
import sys
sys.path.insert(0, '..')

from datetime import date
from pathlib import Path
import polars as pl
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick

from src.pipeline import run, RAW_DIR
from src.ingest.pullpush import fetch_and_save
from src.backtest.metrics import summary

plt.style.use('ggplot')
%matplotlib inline

## Config — edit this cell

In [ ]:
START = date(2025, 1, 1)
END   = date(2025, 12, 31)
HOLD_DAYS   = 5
MOON_PCT    = 0.20
FETCH_DATA  = False  # set True to pull from pullpush.io (takes time)

## 1. Fetch data (skip if already fetched)

In [ ]:
if FETCH_DATA:
    print('Fetching submissions...')
    fetch_and_save(RAW_DIR, START, END, record_type='submissions')
    print('Fetching comments...')
    fetch_and_save(RAW_DIR, START, END, record_type='comments')
else:
    print('Skipping fetch — using cached data in data/raw/')

## 2. Run pipeline

In [ ]:
results, metrics = run(START, END, hold_days=HOLD_DAYS, moon_pct=MOON_PCT)
print(f'Signals with backtest: {len(results)}')

## 3. Summary metrics

In [ ]:
for k, v in metrics.items():
    if isinstance(v, float):
        print(f'  {k:<25} {v:.1%}')
    else:
        print(f'  {k:<25} {v}')

## 4. Signal explorer — top 30 by velocity

In [ ]:
top = (
    results
    .sort('velocity', descending=True)
    .head(30)
    .select(['ticker','date','mentions','velocity','avg_sentiment','forward_return','hit'])
)
top

## 5. Top tickers by signal count, coloured by hit rate

In [ ]:
ticker_stats = (
    results
    .group_by('ticker')
    .agg([
        pl.len().alias('signal_count'),
        pl.col('hit').mean().alias('hit_rate'),
    ])
    .sort('signal_count', descending=True)
    .head(20)
    .to_pandas()
)

fig, ax = plt.subplots(figsize=(12, 5))
colors = ['#2ecc71' if h >= MOON_PCT else '#e74c3c' for h in ticker_stats['hit_rate']]
bars = ax.bar(ticker_stats['ticker'], ticker_stats['signal_count'], color=colors)
ax.set_xlabel('Ticker')
ax.set_ylabel('Signal Count')
ax.set_title(f'Top 20 Tickers by Signal Count  (green = hit rate ≥ {MOON_PCT:.0%})')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 6. Velocity timeline for a single ticker

In [ ]:
TICKER = 'GME'  # change to any ticker

ticker_df = results.filter(pl.col('ticker') == TICKER).sort('date').to_pandas()

if ticker_df.empty:
    print(f'No data for {TICKER}')
else:
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(13, 6), sharex=True)

    ax1.plot(ticker_df['date'], ticker_df['mentions'], label='Mentions')
    ax1.set_ylabel('Daily Mentions')
    ax1.set_title(f'{TICKER} — Mention Velocity & Forward Return')

    ax2.plot(ticker_df['date'], ticker_df['velocity'], color='orange', label='Velocity')
    ax2.axhline(3.0, linestyle='--', color='red', alpha=0.5, label='Signal threshold')
    signal_days = ticker_df[ticker_df['signal']]
    ax2.scatter(signal_days['date'], signal_days['velocity'], color='red', zorder=5, label='Signal fired')
    ax2.set_ylabel('Velocity (x rolling mean)')
    ax2.legend()

    plt.tight_layout()
    plt.show()

## 7. Forward return distribution

In [ ]:
fwd = results.filter(pl.col('forward_return').is_not_null())['forward_return'].to_list()

fig, ax = plt.subplots(figsize=(10, 4))
ax.hist(fwd, bins=40, color='steelblue', edgecolor='white')
ax.axvline(MOON_PCT, color='green', linestyle='--', label=f'Moon threshold ({MOON_PCT:.0%})')
ax.axvline(0, color='black', linestyle='-', alpha=0.3)
ax.xaxis.set_major_formatter(mtick.PercentFormatter(xmax=1))
ax.set_xlabel('5-day Forward Return')
ax.set_ylabel('Signal Count')
ax.set_title('Distribution of Forward Returns on Signals')
ax.legend()
plt.tight_layout()
plt.show()